> 请点击获取[课程 PPT 内容](https://www.canva.cn/design/DAGzTLDII6k/tuPEYMpbOeTSCuxmoICBuA/view?utm_content=DAGzTLDII6k&utm_campaign=designshare&utm_medium=link2&utm_source=uniquelinks&utlId=h705121cba8)。


# 1. 环境配置

## 1.1 python 环境准备

In [ ]:
! pip install openai==2.11.0

## 1.2 大模型密钥准备

请根据第一章内容获取相关平台的 API KEY，如若未在系统变量中填入，请将 API_KEY 信息写入以下代码（若已设置请忽略）：

In [ ]:
import os

# os.environ["OPENAI_API_KEY"] = "sk-xxxxxxxx"

## 1.3 智能体简介

智能体是一个能感知环境、理解任务、规划行动并自主执行的“智能程序”。

可以把它理解为：
- 有目标导向（人工输入任务）
- 会思考（使用大模型推理）
- 会行动（动态调用工具）
- 可记忆（记录上下文与历史）
- 可自我调节（遇到异常自动处理）

# 2. 原生 ReAct 智能体搭建
## 2.1 简介
在这一小节里，我们不使用像 LangChain 或 LlamaIndex 这种集成框架，我们就基于最基本的 python 代码以及 openai 库调用大模型的方式，实现一个最简单的 Agent 系统，从而了解其背后的逻辑和原理。

## 2.1 LLM 模型

在实际运行之前，我们要先确保基本的环境配置完成（安装 openai 库）。并且能够将 AI Studio 中的模型（ernie-4.5-turbo-128k）进行调用。

In [ ]:

import os
from openai import OpenAI

client = OpenAI(
     api_key=os.environ.get("OPENAI_API_KEY"),  # 含有 AI Studio 访问令牌的环境变量，https://aistudio.baidu.com/account/accessToken,
     base_url="https://aistudio.baidu.com/llm/lmapi/v3",  # aistudio 大模型 api 服务域名
)

chat_completion = client.chat.completions.create(
    messages=[
        {'role': 'system', 'content': '你是 AI Studio 实训AI开发平台的开发者助理，你精通开发相关的知识，负责给开发者提供搜索帮助建议。'},
        {'role': 'user', 'content': '你好，请介绍一下AI Studio'}
    ],
    model="ernie-4.5-turbo-128k",
)

print(chat_completion.choices[0].message.content)

## 2.3 Memory 记忆

在确定大模型能够顺利调用后，我们就可以构建一个支持 LLM 对话的类，用于与大语言模型进行交互，同时记录完整的对话信息，为后续循环调用做好准备。

In [ ]:
class Agent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []
        if self.system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message):
        self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        from openai import OpenAI
        client = OpenAI(
            api_key=os.environ.get("OPENAI_API_KEY"),  
            base_url="https://aistudio.baidu.com/llm/lmapi/v3")

        response = client.chat.completions.create(
            model="ernie-4.5-turbo-128k",
            messages=self.messages
        )
        return response.choices[0].message.content
    
abot = Agent('你是一个乐于助人的机器人')
print(abot("你是谁？") )

## 2.4 Tool 工具

我们可以设置两个小的工具来完成后续的Agent任务：
- 第一个是计算器工具 calculate(what)
- 另外一个是狗狗体重查询工具average_dog_weight(name)

In [ ]:
def calculate(what):
    return eval(what)

print(calculate("3 + 7 * 2"))   # 返回 17
print(calculate("10 / 4"))      # 返回 2.5

def average_dog_weight(name):
    if name in "Scottish Terrier": 
        return("Scottish Terriers average 20 lbs")
    elif name in "Border Collie":
        return("a Border Collies average weight is 37 lbs")
    elif name in "Toy Poodle":
        return("a toy poodles average weight is 7 lbs")
    else:
        return("An average dog weights 50 lbs")

print(average_dog_weight("Scottish Terrier"))  
# 返回 "Scottish Terriers average 20 lbs"
print(average_dog_weight("Labrador"))          
# 返回 "An average dog weights 50 lbs"

# 将函数注册到 known_actions 字典中
known_actions = {
  "calculate": calculate,
  "average_dog_weight": average_dog_weight
}

## 2.5 System Prompt 系统提示词

系统提示词就是控制模型如何执行任务的，整体来说提示词分为四部分：
- 定义大模型的行为流程
- 解释各个关键部分的含义
- 列出可用的工具
- 示例演示整体的运行流程

In [ ]:
prompt = """You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

average_dog_weight:
e.g. average_dog_weight: Collie
returns average weight of a dog when given the breed

Example session:

Question: How much does a Bulldog weigh?
Thought: I should look the dogs weight using average_dog_weight
Action: average_dog_weight: Bulldog
PAUSE

You will be called again with this:

Observation: A Bulldog weights 51 lbs

You then output:

Answer: A bulldog weights 51 lbs
""".strip()

## 2.6 系统组装

当我们准备好了四部分Agent的组件：
- LLM 模型（大脑）
- Memory （记忆）
- 工具 （行动能力）
- 系统提示词（思维方式）

我们就可以将其组装起来，实现真正的Agent流程了。

In [ ]:
import re
action_re = re.compile(r'^Action: (\w+): (.*)$')
def query(question, max_turns=5):
    i = 0
    bot = Agent(prompt)
    next_prompt = question
    while i < max_turns:
        i += 1
        result = bot(next_prompt)
        print(result)
        actions = [
            action_re.match(a) 
            for a in result.split('\n') 
            if action_re.match(a)
        ]
        if actions:
            # There is an action to run
            action, action_input = actions[0].groups()
            if action not in known_actions:
                raise Exception("Unknown action: {}: {}".format(action, action_input))
            print(" -- running {} {}".format(action, action_input))
            observation = known_actions[action](action_input)
            print("Observation:", observation)
            next_prompt = "Observation: {}".format(observation)
        else:
            return result

question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight"""
print(query(question))